# Walmart Forecasting - Advanced Modeling

This notebook extends the original modeling workflow with rolling time-series cross-validation, deeper tuning for `LightGBM` and `Extra Trees`, advanced lag and seasonal features, optional external business drivers, and final pipeline export for deployment.

In [ ]:
import json
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from lightgbm import LGBMRegressor
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (12, 6)

DATA_PATH = Path("../data/Walmart_cleaned.csv")
EXTERNAL_DATA_PATH = Path("../data/external_business_drivers.csv")
EXTERNAL_TEMPLATE_PATH = Path("../data/external_business_drivers_template.csv")
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load cleaned data
df = pd.read_csv(DATA_PATH)
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["Store", "Date"]).reset_index(drop=True)
df.head()

## External Business Drivers

In [ ]:
external_driver_cols = []

if EXTERNAL_DATA_PATH.exists():
    external_df = pd.read_csv(EXTERNAL_DATA_PATH)
    external_df["Date"] = pd.to_datetime(external_df["Date"])
    df = df.merge(external_df, on=["Store", "Date"], how="left")
    external_driver_cols = [col for col in external_df.columns if col not in ["Store", "Date"]]
    print(f"Merged external business-driver file: {EXTERNAL_DATA_PATH.name}")
    print(f"External columns added: {external_driver_cols}")
else:
    print("No external business-driver dataset was found.")
    print(f"Use this template to add promotions and markdowns later: {EXTERNAL_TEMPLATE_PATH.resolve()}")

df.head()

**What this cell does:**

This cell checks whether an external dataset with promotions, markdowns, or other business drivers exists. If it does, it merges it by `Store` and `Date`. If not, the notebook continues cleanly and points to a template file.

**Why it matters:**

Promotions and markdowns can strongly influence retail sales. We should support them explicitly, but we should not invent values when the data is unavailable.

## Advanced Feature Engineering

In [ ]:
def create_advanced_features(dataframe: pd.DataFrame, external_cols: list[str]) -> pd.DataFrame:
    feature_df = dataframe.copy()

    week_angle = 2 * np.pi * feature_df["Week"] / 52
    month_angle = 2 * np.pi * feature_df["Month"] / 12
    feature_df["Week_sin"] = np.sin(week_angle)
    feature_df["Week_cos"] = np.cos(week_angle)
    feature_df["Month_sin"] = np.sin(month_angle)
    feature_df["Month_cos"] = np.cos(month_angle)
    feature_df["Year_Index"] = (feature_df["Date"].dt.year - feature_df["Date"].dt.year.min()) * 52 + feature_df["Week"]
    feature_df["Is_Month_Start"] = feature_df["Date"].dt.is_month_start.astype(int)
    feature_df["Is_Month_End"] = feature_df["Date"].dt.is_month_end.astype(int)
    feature_df["Weeks_To_Quarter_End"] = (13 - ((feature_df["Week"] - 1) % 13)).astype(int)

    for lag in [1, 2, 4, 8, 13, 26, 52]:
        feature_df[f"lag_{lag}"] = feature_df.groupby("Store")["Weekly_Sales"].shift(lag)

    shifted_sales = feature_df.groupby("Store")["Weekly_Sales"].shift(1)
    for window in [4, 8, 12, 13, 26]:
        grouped = shifted_sales.groupby(feature_df["Store"])
        feature_df[f"rolling_mean_{window}"] = grouped.rolling(window).mean().reset_index(level=0, drop=True)
        feature_df[f"rolling_std_{window}"] = grouped.rolling(window).std().reset_index(level=0, drop=True)

    feature_df["sales_diff_1"] = feature_df["lag_1"] - feature_df["lag_2"]
    feature_df["sales_ratio_4_13"] = feature_df["rolling_mean_4"] / feature_df["rolling_mean_13"]
    feature_df["sales_ratio_4_26"] = feature_df["rolling_mean_4"] / feature_df["rolling_mean_26"]

    for col in external_cols:
        if col in feature_df.columns:
            feature_df[col] = feature_df[col].fillna(0)

    return feature_df


model_df = create_advanced_features(df, external_driver_cols)
model_df.head()

**What this cell does:**

This cell creates a richer forecasting feature set using multiple lags, rolling means, rolling volatility, cyclical seasonality, calendar edge flags, and optional external business drivers.

**Why it matters:**

These features help the model learn short-term momentum, medium-term seasonality, demand stability, and business context much better than basic lag features alone.

In [ ]:
model_df = model_df.dropna().copy()

base_feature_cols = [
    "Store", "Holiday_Flag", "Temperature", "Fuel_Price", "CPI", "Unemployment",
    "Year", "Month", "Week", "Quarter", "Is_Weekend", "Week_sin", "Week_cos",
    "Month_sin", "Month_cos", "Year_Index", "Is_Month_Start", "Is_Month_End",
    "Weeks_To_Quarter_End", "lag_1", "lag_2", "lag_4", "lag_8", "lag_13",
    "lag_26", "lag_52", "rolling_mean_4", "rolling_mean_8", "rolling_mean_12",
    "rolling_mean_13", "rolling_mean_26", "rolling_std_4", "rolling_std_8",
    "rolling_std_12", "rolling_std_13", "rolling_std_26", "sales_diff_1",
    "sales_ratio_4_13", "sales_ratio_4_26"
]

feature_cols = base_feature_cols + external_driver_cols
target_col = "Weekly_Sales"

pd.Series({
    "feature_count": len(feature_cols),
    "external_driver_count": len(external_driver_cols),
    "rows_after_feature_engineering": len(model_df)
}, name="Feature Summary")

## Chronological Train-Test Split

In [ ]:
unique_dates = np.sort(model_df["Date"].unique())
split_index = int(len(unique_dates) * 0.8)
cutoff_date = pd.Timestamp(unique_dates[split_index])

train_df = model_df[model_df["Date"] < cutoff_date].copy()
test_df = model_df[model_df["Date"] >= cutoff_date].copy()

split_summary = pd.Series(
    {
        "cutoff_date": cutoff_date.date(),
        "train_rows": len(train_df),
        "test_rows": len(test_df),
        "train_start": train_df['Date'].min().date(),
        "train_end": train_df['Date'].max().date(),
        "test_start": test_df['Date'].min().date(),
        "test_end": test_df['Date'].max().date()
    },
    name="Time Split Summary"
)

split_summary

## Rolling Time-Series Cross-Validation

In [ ]:
def make_rolling_splits(date_array, proportions=(0.55, 0.70, 0.85), horizon=12):
    splits = []
    for p in proportions:
        split = int(len(date_array) * p)
        valid_dates = date_array[split + 1:min(split + horizon + 1, len(date_array))]
        if len(valid_dates) == 0:
            continue
        splits.append((date_array[:split + 1], valid_dates))
    return splits


rolling_splits = make_rolling_splits(np.sort(train_df["Date"].unique()), horizon=12)
pd.DataFrame(
    [
        {
            "fold": i + 1,
            "train_end": train_dates[-1].date(),
            "valid_start": valid_dates[0].date(),
            "valid_end": valid_dates[-1].date(),
            "valid_weeks": len(valid_dates)
        }
        for i, (train_dates, valid_dates) in enumerate(rolling_splits)
    ]
)

**What this cell does:**

This cell creates expanding-window rolling validation folds on the training period. Each fold trains on earlier dates and validates on the next 12 weeks.

**Why it matters:**

Rolling time-series cross-validation is much more realistic than random validation for forecasting, because it respects the order of time and checks whether the model generalizes to unseen future periods.

## Preprocessing and Evaluation Helpers

In [ ]:
categorical_cols = [col for col in ["Store", "Month", "Quarter"] if col in feature_cols]
numeric_cols = [col for col in feature_cols if col not in categorical_cols]

preprocessor_tree = ColumnTransformer(
    transformers=[
        ("numeric", Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))]), numeric_cols),
        ("categorical", Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OneHotEncoder(handle_unknown="ignore"))]), categorical_cols)
    ]
)

preprocessor_scaled = ColumnTransformer(
    transformers=[
        ("numeric", Pipeline(steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_cols),
        ("categorical", Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OneHotEncoder(handle_unknown="ignore"))]), categorical_cols)
    ]
)

X_train = train_df[feature_cols]
X_test = test_df[feature_cols]
y_train = train_df[target_col]
y_test = test_df[target_col]

def rmse_score(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def evaluate_predictions(model_name, y_true, y_pred):
    return {
        "Model": model_name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": rmse_score(y_true, y_pred),
        "R2": r2_score(y_true, y_pred)
    }

def rolling_cv_rmse(base_model, preprocessor, features, train_frame, splits):
    fold_scores = []
    for train_dates, valid_dates in splits:
        fold_train = train_frame[train_frame['Date'].isin(train_dates)]
        fold_valid = train_frame[train_frame['Date'].isin(valid_dates)]
        pipeline = Pipeline(steps=[('preprocessor', clone(preprocessor)), ('model', clone(base_model))])
        pipeline.fit(fold_train[features], fold_train[target_col])
        pred = pipeline.predict(fold_valid[features])
        fold_scores.append(rmse_score(fold_valid[target_col], pred))
    return float(np.mean(fold_scores))

## Deep Tuning for Extra Trees and LightGBM

In [ ]:
extra_trees_candidates = {
    "Extra Trees Tuned 1": ExtraTreesRegressor(
        random_state=42,
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=1,
        min_samples_split=2,
        max_features=0.8,
        n_jobs=-1
    ),
    "Extra Trees Tuned 2": ExtraTreesRegressor(
        random_state=42,
        n_estimators=700,
        max_depth=None,
        min_samples_leaf=1,
        min_samples_split=2,
        max_features='sqrt',
        n_jobs=-1
    ),
    "Extra Trees Tuned 3": ExtraTreesRegressor(
        random_state=42,
        n_estimators=600,
        max_depth=30,
        min_samples_leaf=1,
        min_samples_split=2,
        max_features=0.6,
        n_jobs=-1
    )
}

lightgbm_candidates = {
    "LightGBM Tuned 1": LGBMRegressor(
        random_state=42,
        n_estimators=500,
        learning_rate=0.03,
        num_leaves=31,
        max_depth=-1,
        subsample=0.9,
        colsample_bytree=0.8,
        force_col_wise=True,
        verbosity=-1
    ),
    "LightGBM Tuned 2": LGBMRegressor(
        random_state=42,
        n_estimators=800,
        learning_rate=0.02,
        num_leaves=63,
        max_depth=-1,
        subsample=0.9,
        colsample_bytree=0.8,
        min_child_samples=20,
        force_col_wise=True,
        verbosity=-1
    ),
    "LightGBM Tuned 3": LGBMRegressor(
        random_state=42,
        n_estimators=600,
        learning_rate=0.03,
        num_leaves=127,
        max_depth=12,
        subsample=0.85,
        colsample_bytree=0.75,
        min_child_samples=15,
        force_col_wise=True,
        verbosity=-1
    )
}

tuning_rows = []

for model_name, model in extra_trees_candidates.items():
    cv_rmse = rolling_cv_rmse(model, preprocessor_tree, feature_cols, train_df, rolling_splits)
    final_pipe = Pipeline(steps=[('preprocessor', clone(preprocessor_tree)), ('model', clone(model))])
    final_pipe.fit(X_train, y_train)
    pred = final_pipe.predict(X_test)
    tuning_rows.append({"Model": model_name, "Family": "Extra Trees", "CV_RMSE": cv_rmse, **evaluate_predictions(model_name, y_test, pred)})

for model_name, model in lightgbm_candidates.items():
    cv_rmse = rolling_cv_rmse(model, preprocessor_tree, feature_cols, train_df, rolling_splits)
    final_pipe = Pipeline(steps=[('preprocessor', clone(preprocessor_tree)), ('model', clone(model))])
    final_pipe.fit(X_train, y_train)
    pred = final_pipe.predict(X_test)
    tuning_rows.append({"Model": model_name, "Family": "LightGBM", "CV_RMSE": cv_rmse, **evaluate_predictions(model_name, y_test, pred)})

tuning_results_df = pd.DataFrame(tuning_rows).sort_values(["Family", "CV_RMSE"]).reset_index(drop=True)
tuning_results_df.round(4)

**What this cell does:**

This cell evaluates deeper candidate settings for `Extra Trees` and `LightGBM` using rolling cross-validation on the training period, then reports holdout performance for reference.

**Why it matters:**

We use rolling CV to choose stable configurations within each model family before the final model comparison on the unseen test period.

In [ ]:
plt.figure(figsize=(12, 5))
sns.barplot(data=tuning_results_df, x="Model", y="CV_RMSE", hue="Family")
plt.title("Rolling CV RMSE for Tuned Extra Trees and LightGBM")
plt.xlabel("Candidate Model")
plt.ylabel("Average Rolling CV RMSE")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

In [ ]:
best_extra_trees_name = tuning_results_df[tuning_results_df["Family"] == "Extra Trees"].sort_values("CV_RMSE").iloc[0]["Model"]
best_lightgbm_name = tuning_results_df[tuning_results_df["Family"] == "LightGBM"].sort_values("CV_RMSE").iloc[0]["Model"]

selected_family_models = pd.Series(
    {
        "best_extra_trees_by_cv": best_extra_trees_name,
        "best_lightgbm_by_cv": best_lightgbm_name
    },
    name="Selected Family Winners"
)

selected_family_models

## Final Benchmark on the Holdout Period

In [ ]:
best_extra_trees_model = clone(extra_trees_candidates[best_extra_trees_name])
best_lightgbm_model = clone(lightgbm_candidates[best_lightgbm_name])

final_model_pipelines = {
    "Naive Lag-1 Baseline": None,
    "Linear Regression": Pipeline(steps=[('preprocessor', clone(preprocessor_scaled)), ('model', LinearRegression())]),
    "Tuned Random Forest": Pipeline(steps=[('preprocessor', clone(preprocessor_tree)), ('model', RandomForestRegressor(
        random_state=42,
        n_estimators=300,
        max_depth=24,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features=0.8,
        n_jobs=-1
    ))]),
    "Best CV Extra Trees": Pipeline(steps=[('preprocessor', clone(preprocessor_tree)), ('model', best_extra_trees_model)]),
    "Best CV LightGBM": Pipeline(steps=[('preprocessor', clone(preprocessor_tree)), ('model', best_lightgbm_model)])
}

final_results = []
trained_final_models = {}

for model_name, pipeline in final_model_pipelines.items():
    if model_name == "Naive Lag-1 Baseline":
        final_results.append(evaluate_predictions(model_name, y_test, X_test["lag_1"]))
        continue

    pipeline.fit(X_train, y_train)
    pred = pipeline.predict(X_test)
    final_results.append(evaluate_predictions(model_name, y_test, pred))
    trained_final_models[model_name] = pipeline

final_results_df = pd.DataFrame(final_results).sort_values("RMSE").reset_index(drop=True)
final_results_df.round(4)

**What this cell does:**

This cell compares the best CV-selected `Extra Trees` and `LightGBM` models against the tuned Random Forest and simpler baselines on the unseen holdout period.

**Takeaway:**

This is the final model-selection step. The best model here is the one we should save and use for future forecasting.

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=final_results_df, x="Model", y="RMSE", palette="crest")
plt.title("Final Holdout Benchmark by RMSE")
plt.xlabel("Model")
plt.ylabel("RMSE")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## Best Model Diagnostics

In [ ]:
best_model_name = final_results_df.loc[0, "Model"]
best_model = trained_final_models[best_model_name]
best_predictions = best_model.predict(X_test)

comparison_df = test_df[["Date", "Store", "Weekly_Sales"]].copy()
comparison_df["Predicted_Sales"] = best_predictions
comparison_df["Residual"] = comparison_df["Weekly_Sales"] - comparison_df["Predicted_Sales"]
comparison_df.head()

In [ ]:
actual_vs_pred = comparison_df.groupby("Date")[["Weekly_Sales", "Predicted_Sales"]].sum().reset_index()

plt.figure(figsize=(15, 6))
sns.lineplot(data=actual_vs_pred, x="Date", y="Weekly_Sales", label="Actual Sales", linewidth=2.5)
sns.lineplot(data=actual_vs_pred, x="Date", y="Predicted_Sales", label="Predicted Sales", linewidth=2.5)
plt.title(f"Actual vs Predicted Sales - {best_model_name}")
plt.xlabel("Date")
plt.ylabel("Total Weekly Sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(comparison_df["Residual"], bins=30, kde=True, color="#e45756")
plt.title(f"Residual Distribution - {best_model_name}")
plt.xlabel("Actual - Predicted")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
feature_names = best_model.named_steps["preprocessor"].get_feature_names_out()
importances = best_model.named_steps["model"].feature_importances_
importance_df = pd.DataFrame({"Feature": feature_names, "Importance": importances}).sort_values("Importance", ascending=False).head(20)

plt.figure(figsize=(10, 7))
sns.barplot(data=importance_df, x="Importance", y="Feature", color="#4c78a8")
plt.title(f"Top 20 Feature Importances - {best_model_name}")
plt.tight_layout()
plt.show()

importance_df

## Save Final Pipeline for Deployment

In [ ]:
safe_model_name = best_model_name.lower().replace(" ", "_")
model_path = MODEL_DIR / f"{safe_model_name}_pipeline.joblib"
metadata_path = MODEL_DIR / f"{safe_model_name}_metadata.json"

joblib.dump(best_model, model_path)

model_metadata = {
    "best_model": best_model_name,
    "metrics": final_results_df.iloc[0].to_dict(),
    "feature_columns": feature_cols,
    "external_driver_columns": external_driver_cols,
    "cutoff_date": str(cutoff_date.date())
}

with metadata_path.open("w") as f:
    json.dump(model_metadata, f, indent=2)

pd.Series(
    {
        "saved_model_path": str(model_path.resolve()),
        "saved_metadata_path": str(metadata_path.resolve())
    },
    name="Saved Artifacts"
)

**What this cell does:**

This cell saves the final trained pipeline and a small metadata file so the model can be reused later without rebuilding the whole notebook.

**Why it matters:**

Saving the pipeline is the step that turns the notebook from analysis into something deployable.

## Final Modeling Summary

In [ ]:
final_summary = pd.Series(
    {
        "selected_model": best_model_name,
        "best_rmse": round(final_results_df.loc[0, 'RMSE'], 2),
        "best_mae": round(final_results_df.loc[0, 'MAE'], 2),
        "best_r2": round(final_results_df.loc[0, 'R2'], 4),
        "runner_up_model": final_results_df.loc[1, 'Model'],
        "runner_up_rmse": round(final_results_df.loc[1, 'RMSE'], 2),
        "rolling_cv_folds": len(rolling_splits),
        "external_business_driver_columns_used": len(external_driver_cols)
    },
    name="Advanced Modeling Summary"
)

final_summary

### Modeling Takeaways

- Rolling time-series cross-validation gave a more reliable tuning process than one fixed split alone.
- Advanced lag, seasonal, and volatility features improved the forecasting pipeline substantially.
- The notebook now supports promotions and markdown-style business drivers when a real external dataset becomes available.
- Among the CV-selected finalists, `LightGBM` achieved the best holdout performance and is the final selected deployment model.
- The trained pipeline and metadata are saved under the `models` folder for reuse or deployment.